In [1]:
%cd /content
!git clone https://github.com/madelyn-redick/LearningASL.git
%cd /content/LearningASL

/content
Cloning into 'LearningASL'...
remote: Enumerating objects: 41677, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 41677 (delta 23), reused 22 (delta 5), pack-reused 41611 (from 3)
Receiving objects: 100% (41677/41677), 610.84 MiB | 17.21 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Updating files: 100% (42013/42013), done.
/content/LearningASL


In [3]:
# Convert data_preprocessing.ipynb to python module
!jupyter nbconvert --to python data_preprocessing.ipynb --output data_preprocessing

[NbConvertApp] Converting notebook data_preprocessing.ipynb to python
[NbConvertApp] Writing 6358 bytes to data_preprocessing.py


In [5]:
# Convert the CNN.ipynb to a .py file
!jupyter nbconvert --to python cnn/CNN.ipynb

[NbConvertApp] Converting notebook cnn/CNN.ipynb to python
[NbConvertApp] Writing 9436 bytes to cnn/CNN.py


In [6]:
import sys
sys.path.append('/content/LearningASL')
import data_preprocessing

from cnn.CNN import ASL_CNN, train_model, evaluate_model

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

Using Colab cache for faster access to the 'sign-language-dataset-wlasl-videos' dataset.


In [7]:
# Load letter datasets
letter_train = data_preprocessing.letter_train
letter_val = data_preprocessing.letter_val
letter_test = data_preprocessing.letter_test

# Verify imported data
print(f"Type of letter_train: {type(letter_train)}")
print(f"Train: {len(letter_train)}, Val: {len(letter_val)}, Test: {len(letter_test)}")

Type of letter_train: <class 'torchvision.datasets.folder.ImageFolder'>
Train: 33600, Val: 4200, Test: 4200


In [18]:
# Hyperparameter tuning
pretrain_learning_rate = 1e-4 # small learning rate because we are using transfer learning and do not want to mess up pretrained weights
learning_rate = 1e-2 # regular learning rate for randomly initialized layers
momentum = 0.9
lr_gamma = 0.9
epochs = 10
batch_size = 32
print_every = 10

In [19]:
# Setup device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create DataLoader instances for each phase in training
dataloader = {
    'train': DataLoader(letter_train, batch_size=batch_size, shuffle=True, num_workers=2),
    'validation': DataLoader(letter_val, batch_size=batch_size, shuffle=False, num_workers=2)
}

# Create DataLoader for test set
test_dataloader = DataLoader(letter_test, batch_size=batch_size, shuffle=False, num_workers=2)

Using device: cuda:0


In [20]:
# Model Instance
model = ASL_CNN(num_classes=28)

In [21]:
# Setup training
optimizer = optim.SGD([
    {'params': model._base_model.layer4.parameters(), 'lr': pretrain_learning_rate},
    {'params': model._base_model.fc.parameters(), 'lr': learning_rate}
], momentum=momentum)
scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=lr_gamma)
loss_fn = nn.CrossEntropyLoss()

best_model_path = '/content/best_cnn_params.pth'

In [22]:
# Train model
trained_model = train_model(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    dataloader=dataloader,
    device=device,
    best_model_path=best_model_path,
    num_epochs=epochs,
    print_every=print_every
)

Starting training for 10 epochs
Device: cuda:0
Train batches: 1050
Validation batches: 132

Epoch 1/10

Training Phase
  Batch [  10/1050] (  1.0%) | Loss: 3.3257 | Acc: 0.0375
  Batch [  20/1050] (  1.9%) | Loss: 3.3191 | Acc: 0.0594
  Batch [  30/1050] (  2.9%) | Loss: 3.3072 | Acc: 0.0656
  Batch [  40/1050] (  3.8%) | Loss: 3.2877 | Acc: 0.0766
  Batch [  50/1050] (  4.8%) | Loss: 3.2649 | Acc: 0.0944
  Batch [  60/1050] (  5.7%) | Loss: 3.2384 | Acc: 0.1010
  Batch [  70/1050] (  6.7%) | Loss: 3.2041 | Acc: 0.1170
  Batch [  80/1050] (  7.6%) | Loss: 3.1745 | Acc: 0.1281
  Batch [  90/1050] (  8.6%) | Loss: 3.1337 | Acc: 0.1469
  Batch [ 100/1050] (  9.5%) | Loss: 3.0924 | Acc: 0.1619
  Batch [ 110/1050] ( 10.5%) | Loss: 3.0484 | Acc: 0.1790
  Batch [ 120/1050] ( 11.4%) | Loss: 3.0083 | Acc: 0.1924
  Batch [ 130/1050] ( 12.4%) | Loss: 2.9631 | Acc: 0.2055
  Batch [ 140/1050] ( 13.3%) | Loss: 2.9121 | Acc: 0.2172
  Batch [ 150/1050] ( 14.3%) | Loss: 2.8632 | Acc: 0.2321
  Batch [ 1

In [23]:
# Save best model to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_save_path = '/content/drive/MyDrive/LearningASL/models/'
!mkdir -p {drive_save_path}
shutil.copy(best_model_path, f'{drive_save_path}best_cnn_params.pth')
print(f"Model saved to Drive: {drive_save_path}")

Mounted at /content/drive
Model saved to Drive: /content/drive/MyDrive/LearningASL/models/


In [25]:
# Evaluate model with saved best params
from google.colab import drive
drive.mount('/content/drive')

model_path = '/content/drive/MyDrive/LearningASL/models/best_cnn_params.pth'
model.load_state_dict(torch.load(model_path, weights_only=True))

loss, accuracy = evaluate_model(
    model=trained_model,
    loss_fn=loss_fn,
    test_dataloader=test_dataloader,
    device=device,
    print_every=print_every
)

print(f"Loss: {loss}, Accuracy: {accuracy:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Starting Model Evaluation
Batch [  10/ 132] (  7.6%) | Loss: 0.0198 | Acc: 0.9938
Batch [  20/ 132] ( 15.2%) | Loss: 0.0205 | Acc: 0.9938
Batch [  30/ 132] ( 22.7%) | Loss: 0.0557 | Acc: 0.9917
Batch [  40/ 132] ( 30.3%) | Loss: 0.0763 | Acc: 0.9875
Batch [  50/ 132] ( 37.9%) | Loss: 0.0733 | Acc: 0.9863
Batch [  60/ 132] ( 45.5%) | Loss: 0.0694 | Acc: 0.9865
Batch [  70/ 132] ( 53.0%) | Loss: 0.0645 | Acc: 0.9875
Batch [  80/ 132] ( 60.6%) | Loss: 0.0947 | Acc: 0.9859
Batch [  90/ 132] ( 68.2%) | Loss: 0.1547 | Acc: 0.9833
Batch [ 100/ 132] ( 75.8%) | Loss: 0.1475 | Acc: 0.9828
Batch [ 110/ 132] ( 83.3%) | Loss: 0.1414 | Acc: 0.9824
Batch [ 120/ 132] ( 90.9%) | Loss: 0.1350 | Acc: 0.9826
Batch [ 130/ 132] ( 98.5%) | Loss: 0.1281 | Acc: 0.9829
Batch [ 132/ 132] (100.0%) | Loss: 0.1269 | Acc: 0.9831

TEST RESULTS:
Loss: 0.1269 | Accuracy: 0.9831
Evaluation ti